# Chapter 1 Lab: Run-and-Tumble Chemotaxis Simulator
**Basal Cognition — Chapter 1: Intelligence Without a Brain**

In this lab you will build a simulation of *E. coli* run-and-tumble chemotaxis from scratch. The key mechanism is **derivative sensing via methylation-based adaptation**: the cell does not measure absolute glucose concentration — it measures whether concentration is increasing or decreasing compared to a short-term memory baseline.

You will then run an **ablation experiment**: disable the derivative sensing mechanism and observe that navigation fails. This is the computational equivalent of the experiment that revealed *E. coli*'s methylation system is not just a sensor but a temporal comparator.

**Learning objectives:**
- Implement derivative sensing as a running average (methylation baseline)
- Observe how adaptive tumble suppression produces reliable gradient climbing
- Quantify navigation performance across intact vs. ablated conditions
- Apply the Rankin spontaneous-recovery criterion to a simulated agent

**Time:** 45–60 minutes  
**Prerequisites:** Basic Python, NumPy, Matplotlib

In [ ]:
# Install and import
# (numpy and matplotlib are pre-installed on Colab)
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import random
from dataclasses import dataclass, field
from typing import List, Tuple

np.random.seed(42)
print('Imports OK')

## Step 1: Build the Glucose Gradient

We model the chemical environment as a 2D Gaussian concentration field. The food source is at the top-right. Concentration falls off with distance from the peak.

In [ ]:
def make_gradient(size: int = 100, peak_x: int = 90, peak_y: int = 50,
                  sigma: float = 30.0) -> np.ndarray:
    """Create a 2D Gaussian glucose concentration field.
    
    Args:
        size: Grid dimension (size x size)
        peak_x, peak_y: Position of the food source
        sigma: Spatial spread of the gradient
    
    Returns:
        2D numpy array of concentration values in [0, 1]
    """
    y, x = np.mgrid[0:size, 0:size]
    g = np.exp(-((x - peak_x)**2 + (y - peak_y)**2) / (2 * sigma**2))
    return g

GRID = make_gradient(size=100, peak_x=85, peak_y=50, sigma=35)

# Visualize the gradient
plt.figure(figsize=(6, 5))
plt.imshow(GRID, origin='lower', cmap='YlOrRd')
plt.colorbar(label='Glucose concentration')
plt.title('Glucose gradient (food source: top-right)')
plt.xlabel('X position')
plt.ylabel('Y position')
plt.tight_layout()
plt.show()

## Step 2: Implement the Bacterium

The `Bacterium` class models *E. coli*'s chemotaxis machinery:

- **`sense()`** — reads the gradient at the current position (like the Tar receptor)
- **`adapt()`** — updates `methylation_level` toward current concentration (CheR/CheB tug-of-war)
- **`tumble_probability()`** — low when `sense() > methylation_level` (getting better), high when not
- **`step()`** — run or tumble, update position

The `gradient_sensing` flag allows us to ablate the derivative-sensing mechanism.

In [ ]:
@dataclass
class Bacterium:
    """E. coli chemotaxis agent with methylation-based derivative sensing."""
    x: float
    y: float
    angle: float = 0.0          # direction in radians
    methylation_level: float = 0.1  # CheR/CheB methylation baseline
    tau: float = 5.0            # methylation time constant (steps)
    speed: float = 1.5          # run speed (grid units per step)
    gradient_sensing: bool = True  # set False to ablate derivative sensing
    trajectory: List[Tuple[float, float]] = field(default_factory=list)
    
    def sense(self, grid: np.ndarray) -> float:
        """Read glucose concentration at current position."""
        xi = int(np.clip(self.x, 0, grid.shape[1] - 1))
        yi = int(np.clip(self.y, 0, grid.shape[0] - 1))
        return float(grid[yi, xi])
    
    def adapt(self, current_conc: float) -> None:
        """Update methylation level toward current concentration (CheR/CheB dynamics).
        
        This implements the temporal comparator: methylation_level represents
        the recent history, current_conc is now. The difference is the derivative.
        """
        # Exponential moving average — the methylation write register
        alpha = 1.0 / self.tau
        self.methylation_level += alpha * (current_conc - self.methylation_level)
    
    def tumble_probability(self, current_conc: float) -> float:
        """Compute probability of tumbling this step.
        
        If gradient_sensing is disabled, use fixed tumble rate (0.1).
        """
        if not self.gradient_sensing:
            return 0.1  # random walk
        
        # Derivative: is it getting better?
        derivative = current_conc - self.methylation_level
        # High derivative (improving) -> low tumble probability
        # Low or negative derivative (not improving) -> high tumble probability
        p_base = 0.3
        p_tumble = p_base - 2.5 * derivative
        return float(np.clip(p_tumble, 0.02, 0.8))
    
    def step(self, grid: np.ndarray) -> None:
        """Execute one run-or-tumble step."""
        self.trajectory.append((self.x, self.y))
        conc = self.sense(grid)
        p_tumble = self.tumble_probability(conc)
        
        if random.random() < p_tumble:
            # Tumble: pick new random direction
            self.angle = random.uniform(0, 2 * np.pi)
        
        # Run: move in current direction
        new_x = self.x + self.speed * np.cos(self.angle)
        new_y = self.y + self.speed * np.sin(self.angle)
        
        # Bounce off walls
        if 0 <= new_x < grid.shape[1] and 0 <= new_y < grid.shape[0]:
            self.x, self.y = new_x, new_y
        else:
            self.angle = random.uniform(0, 2 * np.pi)  # reflect
        
        # Adapt methylation
        self.adapt(conc)

print('Bacterium class defined.')

## Step 3: Run the Simulation

We simulate 500 steps for an intact bacterium and record its trajectory.

In [ ]:
N_STEPS = 500
START_X, START_Y = 10, 50  # starting position (left center)
FOOD_X, FOOD_Y = 85, 50    # food source position

def run_simulation(gradient_sensing: bool, n_steps: int = N_STEPS,
                   seed: int = 42) -> Bacterium:
    """Run a single bacterium simulation."""
    random.seed(seed)
    np.random.seed(seed)
    b = Bacterium(x=START_X, y=START_Y,
                  angle=random.uniform(0, 2*np.pi),
                  gradient_sensing=gradient_sensing)
    for _ in range(n_steps):
        b.step(GRID)
    return b

# Run intact bacterium
intact = run_simulation(gradient_sensing=True)
print(f'Intact bacterium final position: ({intact.x:.1f}, {intact.y:.1f})')
print(f'Distance to food: {np.sqrt((intact.x-FOOD_X)**2 + (intact.y-FOOD_Y)**2):.1f} grid units')

In [ ]:
def plot_trajectory(bacterium: Bacterium, ax: plt.Axes, color: str,
                    label: str) -> None:
    """Plot a bacterium trajectory on a gradient background."""
    ax.imshow(GRID, origin='lower', cmap='YlOrRd', alpha=0.4,
              extent=[0, 100, 0, 100])
    traj = np.array(bacterium.trajectory)
    ax.plot(traj[:, 0], traj[:, 1], color=color, alpha=0.7, linewidth=0.8)
    ax.plot(traj[0, 0], traj[0, 1], 'ko', markersize=6, label='Start')
    ax.plot(bacterium.x, bacterium.y, 'o', color=color, markersize=8,
            label=f'End ({label})')
    ax.plot(FOOD_X, FOOD_Y, 'g*', markersize=14, label='Food')
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 100)
    ax.set_title(label)
    ax.legend(loc='upper left', fontsize=8)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
plot_trajectory(intact, ax1, color='teal', label='Intact (gradient sensing ON)')
ax1.set_xlabel('X position')
ax1.set_ylabel('Y position')
plt.suptitle('E. coli Run-and-Tumble Chemotaxis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('ch01_intact_trajectory.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ch01_intact_trajectory.png')

## Step 4: The Ablation Experiment

Now run the ablated condition: gradient sensing disabled, fixed tumble rate = 0.1. This simulates a cell with a broken methylation system — it senses but cannot adapt its baseline, so it cannot measure derivatives.

**Prediction:** The ablated bacterium will wander without net drift. The intact bacterium will reliably approach the food source.

In [ ]:
# Run ablated bacterium (same seed for fair comparison)
ablated = run_simulation(gradient_sensing=False, seed=42)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
plot_trajectory(intact, ax1, color='teal', label='Intact (CheR/CheB active)')
ax1.set_xlabel('X position')
ax1.set_ylabel('Y position')

plot_trajectory(ablated, ax2, color='crimson', label='Ablated (CheR/CheB blocked)')
ax2.set_xlabel('X position')

plt.suptitle('Ablation Experiment: Derivative Sensing vs. Random Walk', fontsize=13)
plt.tight_layout()
plt.savefig('ch01_ablation.png', dpi=150, bbox_inches='tight')
plt.show()

d_intact = np.sqrt((intact.x - FOOD_X)**2 + (intact.y - FOOD_Y)**2)
d_ablated = np.sqrt((ablated.x - FOOD_X)**2 + (ablated.y - FOOD_Y)**2)
print(f'Intact final distance to food:  {d_intact:.1f}')
print(f'Ablated final distance to food: {d_ablated:.1f}')
print(f'Improvement ratio: {d_ablated / d_intact:.1f}x better for intact')

## Step 5: Multi-Trial Statistics

A single trial might be lucky. Run 50 trials for each condition and compare the distribution of final distances to the food source.

In [ ]:
N_TRIALS = 50

intact_distances = []
ablated_distances = []

for seed in range(N_TRIALS):
    b_intact = run_simulation(gradient_sensing=True, seed=seed)
    b_ablated = run_simulation(gradient_sensing=False, seed=seed)
    intact_distances.append(
        np.sqrt((b_intact.x - FOOD_X)**2 + (b_intact.y - FOOD_Y)**2))
    ablated_distances.append(
        np.sqrt((b_ablated.x - FOOD_X)**2 + (b_ablated.y - FOOD_Y)**2))

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(intact_distances, bins=15, alpha=0.7, color='teal', label='Intact')
ax.hist(ablated_distances, bins=15, alpha=0.7, color='crimson', label='Ablated')
ax.axvline(np.mean(intact_distances), color='teal', linewidth=2, linestyle='--',
           label=f'Intact mean: {np.mean(intact_distances):.1f}')
ax.axvline(np.mean(ablated_distances), color='crimson', linewidth=2, linestyle='--',
           label=f'Ablated mean: {np.mean(ablated_distances):.1f}')
ax.set_xlabel('Final distance to food source (grid units)')
ax.set_ylabel('Count (n=50 trials each)')
ax.set_title('Derivative Sensing Dramatically Improves Navigation (50 trials)')
ax.legend()
plt.tight_layout()
plt.savefig('ch01_statistics.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Intact:  mean={np.mean(intact_distances):.1f}, std={np.std(intact_distances):.1f}')
print(f'Ablated: mean={np.mean(ablated_distances):.1f}, std={np.std(ablated_distances):.1f}')

## Step 6: Spontaneous Recovery Analog

The Rankin criterion for habituation requires **spontaneous recovery**: after the habituating stimulus stops, the response returns. Here we test the analog: after placing a bacterium in a zero-gradient environment (no food gradient to track), does removing the gradient and then restoring it show that the methylation baseline resets?

We simulate:
1. Normal gradient — bacterium adapts and climbs
2. Flat environment (no gradient) — methylation baseline drifts to zero
3. Gradient restored — does the bacterium re-engage efficiently?

This tests whether the 'memory' in the methylation system is truly volatile (habituation-like) or permanent (structural).

In [ ]:
flat_grid = np.zeros_like(GRID) + 0.01  # nearly uniform concentration

def run_three_phase(n_phase1=200, n_rest=100, n_phase2=200, seed=0):
    """Simulate gradient → flat → gradient to test methylation reset."""
    random.seed(seed)
    np.random.seed(seed)
    
    b = Bacterium(x=10, y=50, gradient_sensing=True)
    distances = []
    phase_labels = []
    
    # Phase 1: normal gradient
    for _ in range(n_phase1):
        b.step(GRID)
        distances.append(np.sqrt((b.x - FOOD_X)**2 + (b.y - FOOD_Y)**2))
        phase_labels.append('Phase 1 (gradient)')
    
    # Phase 2: flat environment (rest/reset)
    for _ in range(n_rest):
        b.step(flat_grid)
        distances.append(np.sqrt((b.x - FOOD_X)**2 + (b.y - FOOD_Y)**2))
        phase_labels.append('Phase 2 (flat/rest)')
    
    # Phase 3: gradient restored
    for _ in range(n_phase2):
        b.step(GRID)
        distances.append(np.sqrt((b.x - FOOD_X)**2 + (b.y - FOOD_Y)**2))
        phase_labels.append('Phase 3 (gradient restored)')
    
    return distances, phase_labels

distances, labels = run_three_phase()

fig, ax = plt.subplots(figsize=(12, 4))
colors = {'Phase 1 (gradient)': 'teal',
          'Phase 2 (flat/rest)': 'gray',
          'Phase 3 (gradient restored)': 'darkorange'}
prev_label = None
for i, (d, lbl) in enumerate(zip(distances, labels)):
    ax.scatter(i, d, color=colors[lbl], s=4, alpha=0.6)

ax.axvline(200, color='black', linestyle='--', alpha=0.4, label='Phase boundaries')
ax.axvline(300, color='black', linestyle='--', alpha=0.4)
patches = [mpatches.Patch(color=c, label=l) for l, c in colors.items()]
ax.legend(handles=patches, loc='upper right', fontsize=8)
ax.set_xlabel('Step')
ax.set_ylabel('Distance to food (grid units)')
ax.set_title('Three-Phase Recovery: Does Methylation Reset?')
plt.tight_layout()
plt.savefig('ch01_recovery.png', dpi=150, bbox_inches='tight')
plt.show()

## Deliverable

Answer the following questions in the cell below. Your responses will be submitted with your notebook.

1. **Ablation result.** What was the mean final distance to food for intact vs. ablated bacteria across 50 trials? What does this tell you about the computational role of the methylation adaptation system?

2. **Mechanism.** In your own words, explain why measuring the *derivative* of concentration (via the methylation baseline) allows better navigation than measuring the *absolute* concentration. Use the bacterium's four-second time constant in your answer.

3. **Recovery.** In the three-phase experiment, does the bacterium re-engage with the gradient in Phase 3 more like Phase 1 (as if the memory reset) or less efficiently (as if the memory persisted)? What does this tell you about the volatility of the methylation write register?

4. **James criterion.** Does your simulated bacterium satisfy William James' criterion of intelligence? Justify your answer using evidence from the ablation experiment. What does the ablation tell you specifically about the 'flexible means' part of the criterion?

5. **Rankin criterion.** Which Rankin criterion does the three-phase experiment most closely test? What additional experiment would you need to run to test *dishabituation* in this system?

**Your answers here:**

1. 

2. 

3. 

4. 

5. 